# Smart Farming — Notebook 3: AI4Bharat Indic-TTS Deployment

Runs **AI4Bharat Indic-TTS alone on port 8004** in a dedicated Google Colab runtime.

- **Models**: Separate language-specific FastPitch acoustic models + HiFi-GAN vocoders for **Indian English (`en`)**, **Gujarati (`gu`)**, and **Hindi (`hi`)**.
- **Hardware Acceleration**: Automatically detects and leverages **CUDA GPU (T4)** for fast ~200ms speech synthesis, while smoothly falling back to CPU if no GPU is available.
- **Persistent Drive Storage**: Checkpoint archives (`en.zip`, `gu.zip`, `hi.zip` ~4.2 GB total) are downloaded and extracted to Google Drive (`MyDrive/SmartFarming/models/indic_tts/checkpoints`) once, and copied to local SSD on startup to eliminate repeated downloads.
- **Cloudflare Integration**: Exposes port 8004 via Cloudflare Quick Tunnel and automatically updates the stable Cloudflare Worker Gateway's KV key `TTS_URL`.

**Prerequisites:** Add Colab Secrets `CF_API_TOKEN`, `CF_ACCOUNT_ID`, `CF_KV_NAMESPACE_ID`, `CF_WORKER_BASE_URL`, and `GATEWAY_API_KEY`.


## 1. Mount Drive and prepare directories


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, time, signal, re, json, requests, shutil
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/SmartFarming"
TTS_SERVER_DIR = "/content/drive/MyDrive/IndicTTSServer"
CHECKPOINTS_DRIVE = os.path.join(DRIVE_ROOT, "models", "indic_tts", "checkpoints")
LOCAL_CHECKPOINTS = "/content/checkpoints"

os.makedirs(TTS_SERVER_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DRIVE, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINTS, exist_ok=True)

print("Drive checkpoints dir:", CHECKPOINTS_DRIVE)
print("Local SSD checkpoints dir:", LOCAL_CHECKPOINTS)


## 2. Checkpoint management (Drive persistence & local SSD cache)

Downloads official AI4Bharat Indic-TTS v1 checkpoints for `en`, `gu`, and `hi` directly into Google Drive if not already present, then copies them to Colab's high-speed local disk.


In [ ]:
LANGUAGES = ["en", "gu", "hi"]
BASE_RELEASE_URL = "https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release"

def verify_model_files(folder):
    required = [
        os.path.join(folder, "fastpitch", "best_model.pth"),
        os.path.join(folder, "fastpitch", "config.json"),
        os.path.join(folder, "hifigan", "best_model.pth"),
        os.path.join(folder, "hifigan", "config.json"),
    ]
    return all(os.path.isfile(p) for p in required)

for lang in LANGUAGES:
    drive_lang_dir = os.path.join(CHECKPOINTS_DRIVE, lang)
    local_lang_dir = os.path.join(LOCAL_CHECKPOINTS, lang)
    
    # Check if Drive has complete model files
    if not verify_model_files(drive_lang_dir):
        print(f"\n[Download] Checkpoints for '{lang}' not found in Google Drive. Downloading from release...")
        zip_url = f"{BASE_RELEASE_URL}/{lang}.zip"
        temp_zip = f"/content/{lang}.zip"
        subprocess.run(["wget", "-c", "--show-progress", zip_url, "-O", temp_zip], check=True)
        print(f"Extracting {lang}.zip to {drive_lang_dir}...")
        os.makedirs(drive_lang_dir, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", temp_zip, "-d", drive_lang_dir], check=True)
        if os.path.isfile(temp_zip):
            os.remove(temp_zip)
        print(f"[Done] Checkpoints for '{lang}' saved to Google Drive.")
    else:
        print(f"[Cached] Checkpoints for '{lang}' found in Google Drive.")
    
    # Fast copy from Drive to local SSD for zero-latency I/O
    if not verify_model_files(local_lang_dir):
        print(f"Copying '{lang}' checkpoints to local SSD {local_lang_dir}...")
        shutil.copytree(drive_lang_dir, local_lang_dir, dirs_exist_ok=True)
    else:
        print(f"Local SSD cache for '{lang}' ready.")

print("\nAll model checkpoints verified and ready on local SSD!")


## 3. Install system and Python dependencies


In [ ]:
# System libraries for audio processing & spellcheck/normalization
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "libsndfile1-dev", "ffmpeg", "enchant-2", "libenchant-2-dev"], check=True)

# FastAPI and server stack
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "requests", "soundfile", "scipy", "pydantic"], check=True)

# Text normalization and transliteration tools
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "indic-numtowords", "aksharamukha", "pyenchant", "pysbd"], check=True)

# Coqui TTS / Indic-TTS synthesis engine
# Pinning compatible release for PyTorch 2.x
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "TTS"], check=True)

print("Indic-TTS dependencies installed successfully.")


## 4. Write Indic-TTS FastAPI server to Drive


In [ ]:
%%writefile /content/drive/MyDrive/IndicTTSServer/indic_tts_server.py
import io
import os
import re
import base64
import threading
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel, Field
from TTS.utils.synthesizer import Synthesizer

# --------------------------------------------------------------------------
# Hardware Configuration
# --------------------------------------------------------------------------
USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"
if not USE_CUDA:
    CPU_THREADS = int(os.getenv("TTS_CPU_THREADS", str(os.cpu_count() or 2)))
    torch.set_num_threads(CPU_THREADS)
print(f"[Init] Indic-TTS starting on device: {DEVICE} (CUDA available: {USE_CUDA})")

# Checkpoint folders
CHECKPOINTS = Path("/content/checkpoints")
LANGUAGES = {
    "en": "Indian English",
    "gu": "Gujarati",
    "hi": "Hindi",
}

app = FastAPI(title="AI4Bharat Indic-TTS Server", version="1.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# --------------------------------------------------------------------------
# Model Management
# --------------------------------------------------------------------------
# If GPU is available with >=12GB VRAM, pre-load models into VRAM.
# If on CPU, load on-demand with thread lock.
_models = {}
_model_lock = threading.Lock()

def create_synthesizer(lang: str) -> Synthesizer:
    folder = CHECKPOINTS / lang
    acoustic = folder / "fastpitch"
    vocoder = folder / "hifigan"
    speaker_file = acoustic / "speakers.pth"
    
    return Synthesizer(
        tts_checkpoint=str(acoustic / "best_model.pth"),
        tts_config_path=str(acoustic / "config.json"),
        tts_speakers_file=(str(speaker_file) if speaker_file.exists() else None),
        tts_languages_file=None,
        vocoder_checkpoint=str(vocoder / "best_model.pth"),
        vocoder_config=str(vocoder / "config.json"),
        encoder_checkpoint="",
        encoder_config="",
        use_cuda=USE_CUDA,
    )

def get_model(lang: str) -> Synthesizer:
    global _models
    if lang in _models:
        return _models[lang]
    
    # If on CPU, keep at most 1 model in memory
    if not USE_CUDA and len(_models) >= 1:
        _models.clear()
        import gc
        gc.collect()
    
    print(f"[Loader] Loading model for '{lang}' on {DEVICE}...")
    synthesizer = create_synthesizer(lang)
    _models[lang] = synthesizer
    print(f"[Loader] Model for '{lang}' loaded successfully.")
    return synthesizer

# Warm up if on GPU
if USE_CUDA:
    for lang in LANGUAGES:
        try:
            get_model(lang)
        except Exception as e:
            print(f"[Warmup Error] Could not pre-load {lang}: {e}")

# --------------------------------------------------------------------------
# Text Preprocessing
# --------------------------------------------------------------------------
def normalize_text_for_tts(text: str, lang: str) -> str:
    clean = text.strip()
    # Replace excessive whitespaces and non-standard punctuation
    clean = re.sub(r"\s+", " ", clean)
    clean = clean.replace("\u200b", "") # Zero-width space
    return clean

# --------------------------------------------------------------------------
# API Models & Endpoints
# --------------------------------------------------------------------------
class TTSRequest(BaseModel):
    text: str = Field(min_length=1, max_length=2000)
    language: str = Field(default="en", pattern="^(en|gu|hi)$")

@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "ai4bharat-indic-tts",
        "device": DEVICE,
        "languages": LANGUAGES,
        "loaded_models": list(_models.keys()),
    }

def synthesize_audio(text: str, lang: str) -> tuple[bytes, int]:
    cleaned = normalize_text_for_tts(text, lang)
    if not cleaned:
        raise ValueError("Text cannot be empty")
    
    with _model_lock:
        model = get_model(lang)
        audio = model.tts(cleaned)
        if audio is None or len(audio) == 0:
            raise RuntimeError("Model returned empty audio waveform")
        
        audio_np = np.asarray(audio, dtype=np.float32)
        audio_np = np.nan_to_num(audio_np)
        sr = model.output_sample_rate or 22050
        
        buf = io.BytesIO()
        sf.write(buf, audio_np, sr, format="WAV", subtype="PCM_16")
        return buf.getvalue(), sr

@app.post("/tts")
def generate_speech_binary(request: TTSRequest):
    """Returns raw WAV audio binary for direct playback or download."""
    try:
        wav_bytes, _ = synthesize_audio(request.text, request.language)
        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers={"Content-Disposition": f'attachment; filename="speech_{request.language}.wav"'},
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

@app.post("/tts/synthesize")
@app.post("/synthesize")
def generate_speech_base64(request: TTSRequest):
    """Returns JSON payload with base64 encoded audio for microservice consumers."""
    try:
        wav_bytes, sr = synthesize_audio(request.text, request.language)
        b64 = base64.b64encode(wav_bytes).decode("utf-8")
        return {
            "audioContent": b64,
            "language": request.language,
            "sampleRate": sr,
            "format": "wav",
            "cached": False,
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc


## 5. Install Cloudflare Tunnel


In [ ]:
cloudflared_path = "/content/cloudflared"
if not os.path.isfile(cloudflared_path):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cloudflared_path], check=True)
os.chmod(cloudflared_path, 0o755)
subprocess.run([cloudflared_path, "--version"], check=True)


## 6. Start Indic-TTS service on port 8004


In [ ]:
def stop_matching(pattern):
    out = subprocess.run(["bash", "-lc", f"pgrep -f {pattern!r} || true"],
                         capture_output=True, text=True).stdout.split()
    for raw_pid in out:
        try:
            pid = int(raw_pid)
            if pid != os.getpid():
                os.kill(pid, signal.SIGTERM)
        except Exception:
            pass

stop_matching("uvicorn indic_tts_server:app")
time.sleep(2)

tts_log = "/content/indic_tts.log"
tts_log_handle = open(tts_log, "w")
tts_proc = subprocess.Popen(
    ["uvicorn", "indic_tts_server:app", "--host", "0.0.0.0", "--port", "8004"],
    cwd=TTS_SERVER_DIR, stdout=tts_log_handle, stderr=subprocess.STDOUT
)

# Wait for service readiness
start_time = time.time()
ready = False
print("Starting Indic-TTS service...")
while time.time() - start_time < 300:
    if tts_proc.poll() is not None:
        print("TTS process exited prematurely:", tts_proc.returncode)
        print(open(tts_log, errors="replace").read()[-4000:])
        break
    try:
        r = requests.get("http://127.0.0.1:8004/health", timeout=3)
        if r.status_code == 200:
            print(f"Indic-TTS service ready in {time.time()-start_time:.1f}s:", r.json())
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(3)

assert ready, "Indic-TTS failed to start. Inspect /content/indic_tts.log."


## 7. Create Quick Tunnel and update Cloudflare KV (`TTS_URL`)


In [ ]:
def open_tunnel(port, timeout=90):
    log_path = f"/content/tunnel_{port}.log"
    log_handle = open(log_path, "w")
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=log_handle, stderr=subprocess.STDOUT
    )
    deadline = time.time() + timeout
    while time.time() < deadline:
        content = open(log_path, errors="replace").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            return proc, match.group(0), log_path
        if proc.poll() is not None:
            break
        time.sleep(1)
    raise RuntimeError(f"Could not obtain tunnel URL for port {port}. Log: {log_path}")

tts_tunnel_proc, TTS_URL, tts_tunnel_log = open_tunnel(8004)
print("TTS_URL =", TTS_URL)

with open("/content/drive/MyDrive/service_urls_tts.json", "w", encoding="utf-8") as f:
    json.dump({"tts": TTS_URL}, f, indent=2)

import requests
from google.colab import userdata

def colab_secret(name, required=True):
    value = userdata.get(name)
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret {name}. Add it to Secrets and enable access.")
    return value

CF_API_TOKEN = colab_secret("CF_API_TOKEN")
CF_ACCOUNT_ID = colab_secret("CF_ACCOUNT_ID")
CF_KV_NAMESPACE_ID = colab_secret("CF_KV_NAMESPACE_ID")

def update_cloudflare_kv(key, value):
    """Update Cloudflare Worker KV storage key."""
    endpoint = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/storage/kv/namespaces/{CF_KV_NAMESPACE_ID}/values/{key}"
    response = requests.put(
        endpoint,
        headers={"Authorization": f"Bearer {CF_API_TOKEN}", "Content-Type": "text/plain"},
        data=value.encode("utf-8"),
        timeout=30
    )
    try:
        payload = response.json()
    except Exception:
        payload = {}
    if not response.ok or payload.get("success") is False:
        raise RuntimeError(f"Cloudflare KV update failed for {key}: {response.text[:400]}")
    print(f"[Cloudflare KV] Updated {key} -> {value}")

update_cloudflare_kv("TTS_URL", TTS_URL)
print("\nStable gateway base URL:", colab_secret("CF_WORKER_BASE_URL"))
print("TTS route: /tts/tts and /tts/synthesize (Authorization: Bearer GATEWAY_API_KEY)")


## 8. Test speech synthesis for Gujarati, Hindi, and English

Generates speech through the stable Cloudflare Worker Gateway and verifies audio output directly inside the notebook.


In [ ]:
import IPython.display as ipd
from google.colab import userdata

GATEWAY_BASE = (userdata.get("CF_WORKER_BASE_URL") or "").rstrip("/")
GATEWAY_KEY = userdata.get("GATEWAY_API_KEY") or ""
assert GATEWAY_BASE and GATEWAY_KEY, "Colab Secrets CF_WORKER_BASE_URL and GATEWAY_API_KEY required."
auth_headers = {"Authorization": f"Bearer {GATEWAY_KEY}", "Content-Type": "application/json"}

test_cases = [
    ("gu", "નમસ્તે ખેડૂત મિત્ર! સ્માર્ટ ફાર્મિંગમાં તમારું સ્વાગત છે. તમારા પાકનું આરોગ્ય સારું છે."),
    ("hi", "नमस्ते किसान भाई! स्मार्ट फार्मिंग में आपका स्वागत है। आपकी फसल स्वस्थ है।"),
    ("en", "Hello farmer! Welcome to Smart Farming. Your crop health looks great today."),
]

for lang, text in test_cases:
    print(f"\n[Synthesizing {lang.upper()}] Text: {text}")
    start = time.time()
    resp = requests.post(
        f"{GATEWAY_BASE}/tts/tts",
        headers=auth_headers,
        json={"text": text, "language": lang},
        timeout=60
    )
    elapsed = time.time() - start
    if resp.status_code == 200:
        print(f"Success in {elapsed:.2f}s! Audio size: {len(resp.content)} bytes")
        out_wav = f"/content/output_{lang}.wav"
        with open(out_wav, "wb") as f:
            f.write(resp.content)
        print(f"Audio preview for {lang}:")
        ipd.display(ipd.Audio(resp.content, rate=22050, autoplay=False))
    else:
        print(f"Failed ({resp.status_code}): {resp.text}")


## 9. Diagnostics


In [ ]:
for log_path in ["/content/indic_tts.log", "/content/tunnel_8004.log"]:
    print("\n====", log_path, "====")
    if os.path.isfile(log_path):
        print(open(log_path, errors="replace").read()[-3000:])
    else:
        print("Not found")
print("\nGPU Status:")
subprocess.run(["bash", "-lc", "nvidia-smi || echo 'Running in CPU mode.'"])
